## Before you start: nothing to build, nothing to install
You are the **auditor**, not the model builder. The model owner's files (model card, lineage document, test data and model) and an independent set of applications are provided for you. This first step only fetches them into this session.

Run the two cells below once (click the play button on each, or press Shift+Enter). Then continue to the audit.

In [ ]:
%pip install -q scikit-learn==1.9.1

In [ ]:
import urllib.request
from pathlib import Path

BASE = "https://raw.githubusercontent.com/AIauditorhub/AIauditorhub/main"
FILES = [
    "management/model_card_v1_3.md",
    "management/lineage_document_v1_3.csv",
    "management/management_training_data.csv",
    "management/management_test_data.csv",
    "management/model_v1_3.pkl",
    "data/independent_applications_m19_m24.csv",
]
if "YOUR" + "-ACCOUNT" in BASE:
    raise SystemExit("The lab files location has not been set. Ask the notebook owner for the corrected link.")
for f in FILES:
    dest = Path(f); dest.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(BASE.rstrip("/") + "/" + f, dest)
import sklearn
print("Files ready:", len(FILES))
print("scikit-learn", sklearn.__version__, "(expected 1.9.1)")
if sklearn.__version__ != "1.9.1":
    print("Version differs: choose Runtime > Restart session, then run this cell again.")

# Episode 1: Is the number true?
**Auditor's notebook. Model: Personal Loan Approval v1.3. All data is synthetic.**

You are the internal auditor. Management's model card reports an AUC and a default rate among approved loans.
Your question: *can the committee rely on the reported performance?*

Work top to bottom. Pause at each **Your turn** and write down your answer before you run the next cell.

In [ ]:
import json, pickle, re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path(".")
FEATURES = ["stated_income","employment_years","credit_bureau_score",
            "debt_to_income","loan_amount","collections_contact_flag"]
CUTOFF = 0.80                     # approve when predicted probability of repayment >= 0.80

# Tolerances are set BEFORE testing. Starting conventions, not standards.
TOL_REPRODUCE = 0.01              # 1 percentage point of AUC
TOL_SHORTFALL = 0.05              # 5 points on independent data
print("Tolerances set: reproduce within", TOL_REPRODUCE, "| independent shortfall within", TOL_SHORTFALL)

## 1. What management reported
Read the model card. This is the **assertion**.

In [ ]:
card = (ROOT/"management"/"model_card_v1_3.md").read_text()
reported_auc = float(re.search(r"AUC \(evaluation data\) \| ([0-9.]+)", card).group(1))
reported_default = float(re.search(r"approved loans \| ([0-9.]+)%", card).group(1))/100
print("Reported AUC:", reported_auc)
print("Reported default rate among auto-approved loans:", f"{reported_default:.1%}")

## 2. Read the lineage document (evidence for one question)
**Question: could each input have been known at the moment of the decision?**
Look at the *captured* column.

In [ ]:
lineage = pd.read_csv(ROOT/"management"/"lineage_document_v1_3.csv")
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 200)
print(lineage[["input","source_system","captured"]].to_string(index=False))
print("\nSplit method recorded:", lineage["split_method"].iloc[0])
late = lineage[lineage["captured"] != "At application"]
print("\nInputs NOT captured at application:", list(late["input"]))

**Your turn.** Which input does not belong, and why does it matter? Write one sentence.

## 3. Check independence by record
Is any application in both the training data and management's test data?

In [ ]:
train = pd.read_csv(ROOT/"management"/"management_training_data.csv")
test  = pd.read_csv(ROOT/"management"/"management_test_data.csv")
overlap = set(train.application_id) & set(test.application_id)
print("Records in both training and test:", len(overlap))
print("Training months:", train.application_month.min(), "to", train.application_month.max(),
      "| Test months:", test.application_month.min(), "to", test.application_month.max())
print("\nNote: no record overlap, but both sets come from the same months and the same process.")

## 4. Reperform: reproduce management's AUC
Use management's model and management's test data. Same method they describe.

In [ ]:
with open(ROOT/"management"/"model_v1_3.pkl","rb") as f:
    model = pickle.load(f)

def p_repay(X):
    return 1 - model.predict_proba(X[FEATURES])[:,1]

mgmt_auc = roc_auc_score(1 - test.defaulted_12m, p_repay(test))
diff = abs(mgmt_auc - reported_auc)
print(f"Reported {reported_auc:.3f} | Reperformed {mgmt_auc:.3f} | Difference {diff:.3f}")
print("Within reproduction tolerance:", diff <= TOL_REPRODUCE)

**Your turn.** The number reproduces. Is the assertion now proved? Decide before you continue.

## 5. Reperform on independent data
Applications from months 19 to 24, after the training data was frozen, captured as the feed looked **at the decision date**.

In [ ]:
indep = pd.read_csv(ROOT/"data"/"independent_applications_m19_m24.csv")
y = 1 - indep.defaulted_12m
scores = p_repay(indep)
auc_ind = roc_auc_score(y, scores)

rng = np.random.default_rng(1)
boots = []
for _ in range(500):
    idx = rng.integers(0, len(indep), len(indep))
    boots.append(roc_auc_score(y.iloc[idx], scores[idx]))
lo, hi = np.percentile(boots, [2.5, 97.5])
shortfall = reported_auc - auc_ind
print(f"Independent AUC: {auc_ind:.3f}  (95% CI {lo:.3f} to {hi:.3f}), n = {len(indep)}")
print(f"Shortfall vs reported: {shortfall:.3f}  | tolerance {TOL_SHORTFALL}")
print("Within shortfall tolerance:", shortfall <= TOL_SHORTFALL)

## 6. Test at the cutoff the business actually uses
Among loans the system would auto-approve (score >= 0.80), what is the default rate?

In [ ]:
appr_ind = scores >= CUTOFF
default_ind = indep.loc[appr_ind, "defaulted_12m"].mean()
test_scores = p_repay(test)
appr_mgmt = test_scores >= CUTOFF
default_mgmt = test.loc[appr_mgmt, "defaulted_12m"].mean()
print(f"Approved on management's test data: {appr_mgmt.mean():.1%} of applications, default rate {default_mgmt:.1%}")
print(f"Approved on independent data:       {appr_ind.mean():.1%} of applications, default rate {default_ind:.1%}")
print(f"Reported on the model card:         {reported_default:.1%}")

## 7. Explain the gap
The lineage document pointed at one field. Test the explanation: neutralise it on management's test data, exactly as it will look at decision time, and rescore. You are not retraining anything.

In [ ]:
test_dt = test.copy()
test_dt["collections_contact_flag"] = 0            # what the feed holds at the decision date
auc_mgmt_neutral = roc_auc_score(1 - test_dt.defaulted_12m, p_repay(test_dt))
print(f"Management test AUC as reported:               {mgmt_auc:.3f}")
print(f"Management test AUC with the flag as it is at decision time: {auc_mgmt_neutral:.3f}")
print(f"Independent AUC:                                {auc_ind:.3f}")
print("\nThe gap closes once the late-captured field is removed from the evaluation.")

## 8. Write the finding
Complete the five parts in your own words, then compare with the facilitator version.

| Part | Your wording |
| --- | --- |
| Condition | |
| Criterion | |
| Cause | |
| Effect | |
| Rating | |

In [ ]:
results = {
    "reported_auc": reported_auc,
    "reperformed_auc_mgmt_test": round(float(mgmt_auc), 4),
    "independent_auc": round(float(auc_ind), 4),
    "independent_auc_ci": [round(float(lo), 4), round(float(hi), 4)],
    "shortfall": round(float(shortfall), 4),
    "tolerance_reproduce": TOL_REPRODUCE,
    "tolerance_shortfall": TOL_SHORTFALL,
    "reported_default_among_approved": reported_default,
    "default_among_approved_mgmt_test": round(float(default_mgmt), 4),
    "default_among_approved_independent": round(float(default_ind), 4),
    "approval_rate_independent": round(float(appr_ind.mean()), 4),
    "auc_mgmt_test_flag_neutralised": round(float(auc_mgmt_neutral), 4),
    "record_overlap_train_test": len(overlap),
    "n_independent": int(len(indep)),
    "n_mgmt_test": int(len(test)),
}
Path("results_ep01.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))